# 1. Perkenalan Dataset

Pada proyek ini, digunakan Student Lifestyle and Stress Prediction Dataset yang diperoleh dari Kaggle. Dataset ini berisi informasi mengenai gaya hidup, dukungan sosial, dan karakteristik mahasiswa yang digunakan untuk memprediksi tingkat stres.

Dataset ini memiliki total sekitar 25.500 baris dengan fitur-fitur berikut:
- `Student_Type`: tipe mahasiswa (misalnya `school`, `college`, `working_student`)
- `Sleep_Hours`: jumlah jam tidur per hari
- `Study_Hours`: jumlah jam belajar per hari
- `Social_Media_Hours`: waktu penggunaan media sosial per hari
- `Attendance`: persentase kehadiran kuliah
- `Exam_Pressure`: tingkat tekanan ujian
- `Family_Support`: tingkat dukungan keluarga
- `Month`: bulan pengambilan data

Target pada dataset ini berupa klasifikasi biner (`Stress_Level`) yang menunjukkan apakah mahasiswa mengalami:
- `0`: stres rendah
- `1`: stres tinggi

Dataset dapat diakses melalui tautan berikut:
https://www.kaggle.com/datasets/sridevilavanyacse/student-lifestyle-and-stress-prediction-dataset/data

# **2. Import Library**

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import dagshub

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import collections
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# **3. Memuat Dataset**

In [ ]:
# 1. Dapatkan jalur absolut dari folder proyek utama Anda
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))

# 2. Gabungkan jalur ke berkas data menggunakan os.path.join (Aman untuk Windows/Linux)
DATA_PATH = os.path.join(
    BASE_DIR, "data_raw", "student-lifestyle-and-stress-dataset.csv"
)

# 3. Baca dataset
df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns)

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini dilakukan eksplorasi data untuk memahami karakteristik dataset, distribusi fitur, hubungan antar variabel, serta mendeteksi potensi permasalahan seperti missing value, outlier, dan ketidakseimbangan kelas.

Exploratory Data Analysis (EDA) membantu dalam menentukan strategi preprocessing dan pemilihan model machine learning yang tepat.

In [ ]:
print("Shape Dataset:", df.shape)

df.info()

In [ ]:
df.describe()

In [ ]:
missing_values = df.isnull().sum()

missing_values

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(x='Stress_Level', data=df)

plt.title("Distribusi Target (Stress Level)")
plt.xlabel("Stress Level (0 = Low, 1 = High)")
plt.ylabel("Count")
plt.show()

In [70]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

df = pd.read_csv("Eksperimen_SML_Rifdan/data_raw/student-lifestyle-and-stress-dataset.csv")

X = df.drop(columns=['stress Level']) 
y = df['stress Level']

# 1. Pisahkan data menjadi Train dan Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Lakukan Scaling hanya fit pada X_train untuk mencegah leakage
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Eksekusi Random Undersampling menggunakan X_train yang sudah di-scale
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train_scaled, y_train)

# 3. --- OUTPUT 2: GAMBAR COUNTPLOT PERBANDINGAN (BEBAS WARNING) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot Sebelum Undersampling
sns.countplot(x=y_train, ax=axes[0], hue=y_train, palette="viridis", legend=False)
axes[0].set_title("Sebelum Undersampling (Data Train)")
axes[0].set_xlabel("Tingkat Stres")
axes[0].set_ylabel("Jumlah Sampel")

# Plot Sesudah Undersampling
sns.countplot(x=y_train_resampled, ax=axes[1], hue=y_train_resampled, palette="viridis", legend=False)
axes[1].set_title("Sesudah Undersampling (Data Train)")
axes[1].set_xlabel("Tingkat Stres")
axes[1].set_ylabel("Jumlah Sampel")

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Eksperimen_SML_Rifdan/data_raw/student-lifestyle-and-stress-dataset.csv'

In [ ]:
df.hist(figsize=(15,12))

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. OTOMATIS: Mengambil semua kolom bertipe angka (int/float) kecuali target 'Stress_Level'
# Ini mencegah KeyError akibat salah penulisan huruf besar/kecil
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

if "Stress_Level" in numerical_cols:
    numerical_cols.remove("Stress_Level")

# 2. Atur ukuran grid secara dinamis berdasarkan jumlah kolom numerik yang ditemukan
num_features = len(numerical_cols)
# Membuat grid dengan 3 kolom, baris menyesuaikan
num_rows = (num_features + 2) // 3

fig, axes = plt.subplots(num_rows, 3, figsize=(16, 4 * num_rows))
axes = axes.flatten()

# 3. Loop untuk menggambar boxplot
for i, col in enumerate(numerical_cols):
    clean_data = pd.to_numeric(df[col], errors="coerce")

    sns.boxplot(x=clean_data, ax=axes[i], color="lightgreen")
    axes[i].set_title(f"Boxplot of {col}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("")

# 4. Hapus sisa subplot yang kosong di bagian akhir grid
for j in range(num_features, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    "Deteksi Outlier pada Fitur Numerik Gaya Hidup Mahasiswa",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

Berdasarkan hasil Exploratory Data Analysis (EDA) di notebook saya, diperoleh beberapa insight penting sebagai berikut:

Dataset berisi sekitar 25.500 baris dengan fitur utama berupa kombinasi numerik dan kategorikal: Student_Type, Sleep_Hours, Study_Hours, Social_Media_Hours, Attendance, Exam_Pressure, Family_Support, Month, dan target Stress_Level.
Saya sudah memeriksa missing value dengan df.isnull().sum(), sehingga saya bisa menilai apakah data sudah bersih atau perlu penanganan missing value sebelum preprocessing lebih lanjut.
Distribusi target Stress_Level tidak sepenuhnya seimbang, sehingga model raw cenderung berisiko bias ke kelas mayoritas. Karena itu saya menyiapkan langkah balancing menggunakan teknik resampling.
Visualisasi histogram dan heatmap korelasi membantu mengidentifikasi fitur numerik yang berpotensi penting, terutama Sleep_Hours, Study_Hours, Exam_Pressure, Attendance, dan Family_Support.
Boxplot deteksi outlier menunjukkan adanya nilai ekstrem pada beberapa fitur numerik, yang perlu diperhatikan dan mungkin akan ditangani pada tahap preprocessing berikutnya.
Saya sudah membuat countplot perbandingan sebelum dan sesudah sampling, sehingga bisa memantau secara visual perubahan distribusi kelas yang terjadi setelah penyeimbangan.
Sebagian besar fitur utama sudah berbentuk numerik, sehingga proses scaling dan pelatihan model machine learning akan menjadi lebih mudah setelah preprocessing.
Kesimpulan: EDA saya sudah siap menunjang tahap preprocessing dan modeling berikutnya, terutama dengan fokus pada penyeimbangan kelas dan pemilihan fitur numerik yang relevan.

# **5. Data Preprocessing**

In [ ]:
# Tambahkan errors='ignore' agar aman saat cell dijalankan berulang kali
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Tampilkan 5 data pertama untuk memastikan struktur data sudah bersih
df.head()

In [ ]:
# Sesuai dengan nama kolom target asli pada Student Lifestyle and Stress Dataset
X = df.drop("Stress_Level", axis=1)
y = df["Stress_Level"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. Pisahkan Fitur dan Target dengan benar
X = df.drop("Stress_Level", axis=1)
y = df["Stress_Level"]

# 2. SEBELUM SPLIT: Ubah kolom string ('working_student', dll) menjadi angka biner
X_encoded = pd.get_dummies(X, drop_first=False)

# 3. Drop missing values (aman untuk kualitas data)
mask = ~X_encoded.isnull().any(axis=1)
X_encoded_clean = X_encoded[mask]
y_clean = y[mask]

print(f"Data sebelum drop: {X_encoded.shape}, Sesudah: {X_encoded_clean.shape}")

# 4. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

# 5. Imputasi sebagai backup (strategi median)
imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

# 6. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaling berhasil! Ukuran data latihan:", X_train_scaled.shape)

In [ ]:
# 1. Ubah kembali array NumPy hasil scaling menjadi DataFrame Pandas
# Kita gunakan X_train.columns karena sudah menampung kolom baru hasil get_dummies
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# 2. Tampilkan 5 data pertama untuk verifikasi akhir
print("=== 5 BARIS PERTAMA DATA TRAINING TERSKALAKAN ===")
X_train_scaled.head()

In [ ]:
os.makedirs('data_processed', exist_ok=True)

X_train_scaled.to_csv('data_processed/X_train.csv', index=False)
X_test_scaled.to_csv('data_processed/X_test.csv', index=False)

y_train.to_csv('data_processed/y_train.csv', index=False)
y_test.to_csv('data_processed/y_test.csv', index=False)
     

In [ ]:
import os

print(os.listdir('data_processed'))

# **6. Modelling**

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
}

results = []

# --- LOOPING EVALUASI ---
for name, model in models.items():
    # Latih Model
    model.fit(X_train_scaled, y_train)

    # Prediksi Data Uji
    y_pred = model.predict(X_test_scaled)

    # Hitung Metrik Evaluasi
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="binary")
    recall = recall_score(y_test, y_pred, average="binary")
    f1 = f1_score(y_test, y_pred, average="binary")

    # Simpan Hasil
    results.append(
        {
            "Model": name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
        }
    )

# Tampilkan ringkasan performa dalam bentuk DataFrame
results_df = pd.DataFrame(results)
results_df